In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load and preprocess data
train = pd.read_csv(dataset_root / "AutoEncoder/Sign Language MNIST/sign_mnist_train.csv")
test = pd.read_csv(dataset_root / "AutoEncoder/Sign Language MNIST/sign_mnist_test.csv")

print("Training CSV shape:", train.shape)
print("Test CSV shape:", test.shape)
print("First three training rows:")
print(train.head(3))
print("Training label counts:")
print(train.label.value_counts().sort_index())
print("Y images excluded from training:", (train.label == 24).sum())
print("Y images kept in testing:", (test.label == 24).sum())

x_train = train[train.label != 24].drop(columns="label").to_numpy("float32") / 255
x_test = test.drop(columns="label").to_numpy("float32") / 255

print("Training input shape:", x_train.shape)
print("Test input shape:", x_test.shape)
print("Training value range:", x_train.min(), "to", x_train.max())
print("Test value range:", x_test.min(), "to", x_test.max())
print("First training signal: first ten feature values")
print(x_train[0, :10])


In [ ]:
# Create Autoencoder
model = Sequential([
    Input(shape=(784,)),
    Dense(32, activation='relu'),
    Dense(784, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='mse'
)
model.summary()


In [ ]:
# Train using normal inputs as their own reconstruction targets
history = model.fit(x_train, x_train, epochs=10, batch_size=128)
print("Last three training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# Reconstruction errors for normal training inputs
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=1)
print("First five training errors:", train_error[:5])
print("Training error min / mean / max:", train_error.min(), train_error.mean(), train_error.max())
threshold = np.percentile(train_error, 95)
print("Anomaly threshold (95th percentile):", threshold)


In [ ]:
# Reconstruction errors for test inputs
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=1)
anomalies = test_error > threshold
print("First ten test errors:", test_error[:10])
print("Test error min / mean / max:", test_error.min(), test_error.mean(), test_error.max())
print("First ten detected flags:", anomalies[:10])
print("Anomalies:", anomalies.sum())
print("Y Detected:", (anomalies & (test.label == 24)).sum())
print("First ten test labels and anomaly decisions:")
print(pd.DataFrame({
    "Actual label": test.label.to_numpy()[:10],
    "Actual Y": test.label.to_numpy()[:10] == 24,
    "Detected anomaly": anomalies[:10],
    "Error": test_error[:10]
}))
